# Extract Range of Motion Data from Archived Results

This notebook extracts minimum and maximum muscle lengths from all simulation folders in `archived_results/` and compiles them into a single CSV file.

In [2]:
import os
import re
import pandas as pd
from pathlib import Path

In [3]:
# Define the base directory
base_dir = Path('archived_results')
output_file = 'rom_data_all_combinations.csv'

# Check if archived_results directory exists
if not base_dir.exists():
    raise FileNotFoundError(f"Directory {base_dir} not found")

print(f"Base directory: {base_dir.absolute()}")
print(f"Output file: {output_file}")

Base directory: /Users/canis/Library/CloudStorage/OneDrive-Persönlich/work/SdV/natwissKolleg/results/final_results/results_for_paper/da_4/archived_results
Output file: rom_data_all_combinations.csv


In [4]:
def parse_folder_name(folder_name):
    """
    Parse folder name to extract parameters.
    Format: F{force}_y{muscle_y}_Vol{volume}_Am{am}_rho{rho}
    Example: F00_y32.69_Vol421.6_Am450_rho10.493
    """
    pattern = r'F(\d+)_y([\d.]+)_Vol([\d.]+)_Am(\d+)_rho([\d.]+)'
    match = re.match(pattern, folder_name)
    
    if match:
        return {
            'force_N': float(match.group(1)),
            'muscle_extent_y_cm': float(match.group(2)),
            'volume_cm3': float(match.group(3)),
            'am_cm_inv': float(match.group(4)),
            'rho_1e4_kg_cm3': float(match.group(5))
        }
    else:
        return None

def extract_rom_data(rom_file_path):
    """
    Extract min and max muscle length from range_of_motion.txt file.
    """
    try:
        with open(rom_file_path, 'r') as f:
            content = f.read()
        
        # Extract z_max
        z_max_match = re.search(r'Maximum muscle length \(z_max\):\s+([\d.]+)\s+cm', content)
        # Extract z_min
        z_min_match = re.search(r'Minimum muscle length \(z_min\):\s+([\d.]+)\s+cm', content)
        # Extract ROM
        rom_match = re.search(r'Range of Motion \(ROM\):\s+([\d.]+)\s+cm', content)
        
        if z_max_match and z_min_match and rom_match:
            return {
                'z_max_cm': float(z_max_match.group(1)),
                'z_min_cm': float(z_min_match.group(1)),
                'rom_cm': float(rom_match.group(1))
            }
        else:
            return None
    except Exception as e:
        print(f"Error reading {rom_file_path}: {e}")
        return None

# Test parsing on a sample folder name
test_folder = "F00_y32.69_Vol421.6_Am450_rho10.493"
print(f"Test parsing: {test_folder}")
print(parse_folder_name(test_folder))

Test parsing: F00_y32.69_Vol421.6_Am450_rho10.493
{'force_N': 0.0, 'muscle_extent_y_cm': 32.69, 'volume_cm3': 421.6, 'am_cm_inv': 450.0, 'rho_1e4_kg_cm3': 10.493}


In [5]:
# Iterate through all folders in archived_results
data_list = []
folders_processed = 0
folders_failed = 0

for folder in sorted(base_dir.iterdir()):
    if folder.is_dir():
        # Parse folder name to get parameters
        params = parse_folder_name(folder.name)
        
        if params is None:
            print(f"Warning: Could not parse folder name: {folder.name}")
            folders_failed += 1
            continue
        
        # Look for range_of_motion.txt file
        rom_file = folder / 'range_of_motion.txt'
        
        if rom_file.exists():
            rom_data = extract_rom_data(rom_file)
            
            if rom_data is not None:
                # Combine parameters and ROM data
                row_data = {**params, **rom_data}
                row_data['folder_name'] = folder.name
                data_list.append(row_data)
                folders_processed += 1
            else:
                print(f"Warning: Could not extract ROM data from: {rom_file}")
                folders_failed += 1
        else:
            print(f"Warning: range_of_motion.txt not found in: {folder.name}")
            folders_failed += 1

print(f"\nFolders processed successfully: {folders_processed}")
print(f"Folders failed: {folders_failed}")


Folders processed successfully: 5040
Folders failed: 0


In [6]:
# Create DataFrame
df = pd.DataFrame(data_list)

# Reorder columns for better readability
column_order = [
    'folder_name',
    'force_N',
    'muscle_extent_y_cm',
    'volume_cm3',
    'am_cm_inv',
    'rho_1e4_kg_cm3',
    'z_max_cm',
    'z_min_cm',
    'rom_cm'
]

df = df[column_order]

# Display summary
print(f"\nTotal rows: {len(df)}")
print(f"\nFirst few rows:")
df.head(10)


Total rows: 5040

First few rows:


,folder_name,force_N,muscle_extent_y_cm,volume_cm3,am_cm_inv,rho_1e4_kg_cm3,z_max_cm,z_min_cm,rom_cm
0,F00_y32.69_Vol421.6_Am450_rho10.493,0.0,32.69,421.6,450.0,10.493,32.69,29.390991,3.299010
1,F00_y32.69_Vol421.6_Am450_rho10.534,0.0,32.69,421.6,450.0,10.534,32.69,29.392965,3.297035
2,F00_y32.69_Vol421.6_Am450_rho10.575,0.0,32.69,421.6,450.0,10.575,32.69,29.394968,3.295032
3,F00_y32.69_Vol421.6_Am450_rho10.616,0.0,32.69,421.6,450.0,10.616,32.69,29.396904,3.293096
4,F00_y32.69_Vol421.6_Am500_rho10.493,0.0,32.69,421.6,500.0,10.493,32.69,29.383631,3.306369
5,F00_y32.69_Vol421.6_Am500_rho10.534,0.0,32.69,421.6,500.0,10.534,32.69,29.385613,3.304387
6,F00_y32.69_Vol421.6_Am500_rho10.575,0.0,32.69,421.6,500.0,10.575,32.69,29.387517,3.302483
7,F00_y32.69_Vol421.6_Am500_rho10.616,0.0,32.69,421.6,500.0,10.616,32.69,29.389461,3.300540
8,F00_y32.69_Vol421.6_Am550_rho10.493,0.0,32.69,421.6,550.0,10.493,32.69,29.376303,3.313698
9,F00_y32.69_Vol421.6_Am550_rho10.534,0.0,32.69,421.6,550.0,10.534,32.69,29.378201,3.311799


In [7]:
# Display summary statistics
print("\nParameter ranges:")
print(f"Force: {df['force_N'].min()} - {df['force_N'].max()} N")
print(f"Muscle extent y: {df['muscle_extent_y_cm'].min()} - {df['muscle_extent_y_cm'].max()} cm")
print(f"Volume: {df['volume_cm3'].min()} - {df['volume_cm3'].max()} cm³")
print(f"Am: {df['am_cm_inv'].min()} - {df['am_cm_inv'].max()} cm⁻¹")
print(f"Rho: {df['rho_1e4_kg_cm3'].min()} - {df['rho_1e4_kg_cm3'].max()} 1e-4 kg/cm³")
print(f"\nROM ranges:")
print(f"z_max: {df['z_max_cm'].min()} - {df['z_max_cm'].max()} cm")
print(f"z_min: {df['z_min_cm'].min()} - {df['z_min_cm'].max()} cm")
print(f"ROM: {df['rom_cm'].min()} - {df['rom_cm'].max()} cm")

df.describe()


Parameter ranges:
Force: 0.0 - 31.0 N
Muscle extent y: 32.69 - 34.97 cm
Volume: 421.6 - 738.0 cm³
Am: 450.0 - 550.0 cm⁻¹
Rho: 10.493 - 10.616 1e-4 kg/cm³

ROM ranges:
z_max: 32.69000003 - 49.9403764 cm
z_min: 29.37630262 - 48.97428965 cm
ROM: 0.0 - 3.95466747 cm


,force_N,muscle_extent_y_cm,volume_cm3,am_cm_inv,rho_1e4_kg_cm3,z_max_cm,z_min_cm,rom_cm
count,5040.000000,5040.000000,5040.000000,5040.00000,5040.000000,5040.000000,5040.000000,5040.000000
mean,16.333333,33.746000,575.314286,500.00000,10.554500,37.961135,34.809399,3.151736
std,10.103257,0.790359,110.016596,40.82888,0.045844,3.488735,3.578798,0.546532
min,0.000000,32.690000,421.600000,450.00000,10.493000,32.690000,29.376303,0.000000
25%,8.250000,33.190000,455.900000,450.00000,10.523750,35.162669,32.157001,2.813023
50%,16.500000,33.690000,573.500000,500.00000,10.554500,37.334106,33.969040,3.290646
75%,24.750000,34.190000,691.200000,550.00000,10.585250,40.097394,36.531541,3.560178
max,31.000000,34.970000,738.000000,550.00000,10.616000,49.940376,48.974290,3.954667


In [8]:
# Save to CSV
df.to_csv(output_file, index=False)
print(f"\nData saved to: {output_file}")
print(f"Absolute path: {Path(output_file).absolute()}")


Data saved to: rom_data_all_combinations.csv
Absolute path: /Users/canis/Library/CloudStorage/OneDrive-Persönlich/work/SdV/natwissKolleg/results/final_results/results_for_paper/da_4/rom_data_all_combinations.csv


In [9]:
# Verify unique combinations
print("\nUnique values per parameter:")
print(f"Forces: {sorted(df['force_N'].unique())}")
print(f"Muscle Y: {sorted(df['muscle_extent_y_cm'].unique())}")
print(f"Volumes: {sorted(df['volume_cm3'].unique())}")
print(f"Am values: {sorted(df['am_cm_inv'].unique())}")
print(f"Rho values: {sorted(df['rho_1e4_kg_cm3'].unique())}")


Unique values per parameter:
Forces: [np.float64(0.0), np.float64(3.0), np.float64(6.0), np.float64(9.0), np.float64(12.0), np.float64(15.0), np.float64(18.0), np.float64(21.0), np.float64(24.0), np.float64(27.0), np.float64(30.0), np.float64(31.0)]
Muscle Y: [np.float64(32.69), np.float64(33.19), np.float64(33.69), np.float64(34.19), np.float64(34.97)]
Volumes: [np.float64(421.6), np.float64(455.9), np.float64(514.7), np.float64(573.5), np.float64(632.3), np.float64(691.2), np.float64(738.0)]
Am values: [np.float64(450.0), np.float64(500.0), np.float64(550.0)]
Rho values: [np.float64(10.493), np.float64(10.534), np.float64(10.575), np.float64(10.616)]


In [ ]:
# Group by muscle_extent_y_cm and volume_cm3, and find min/max for each ROM metric
grouped = df.groupby(['muscle_extent_y_cm', 'volume_cm3']).agg({
    'z_max_cm': ['min', 'max'],
    'z_min_cm': ['min', 'max'],
    'rom_cm': ['min', 'max']
}).round(6)

# Flatten column names
grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]

print(f"Min/Max values for each muscle_extent_y_cm and volume_cm3 combination:")
print(f"Total combinations: {len(grouped)}\n")
print(grouped)


Min/Max values for each muscle_extent_y_cm and volume_cm3 combination:
Total combinations: 35

                               z_max_cm_min  z_max_cm_max  z_min_cm_min  \
muscle_extent_y_cm volume_cm3                                             
32.69              421.6          32.690000     45.244163     29.376303   
                   455.9          32.690000     43.759133     29.582858   
                   514.7          32.690000     41.839325     29.881586   
                   573.5          32.690000     40.464917     30.127733   
                   632.3          32.690000     39.442884     30.331514   
                   691.2          32.690000     38.655041     30.501316   
                   738.0          32.690000     38.148473     30.615992   
33.19              421.6          33.190000     46.252569     29.761098   
                   455.9          33.190000     44.704998     29.975528   
                   514.7          33.190000     42.699211     30.281473   
     